In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\shafi\number-simplex-reproduction\notebooks1


In [4]:
import sys
from pathlib import Path

cwd = Path.cwd()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise RuntimeError(
        "Could not find the src folder. "
        f"Current directory is: {cwd}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: c:\Users\shafi\number-simplex-reproduction
src exists: True


In [5]:
from src.firing_rate_tuning import analyze_neuron
from src.number_decoding_analysis import make_temporal_features
from src.poisson_naive_bayes import cv_poisson_nb

In [6]:
result = analyze_neuron(
    "YFU",
    43,
    root=PROJECT_ROOT,
    plot=False,
    verbose=True,
)

Subject: YFU

Total recorded neurons: 78

All recorded regions:
  acc: 16
  amy: 23
  hpc: 39

MTL regions used in paper:
  amy: 23
  hpc: 39

Total MTL neurons: 62

Selected neuron:
  MATLAB row: 43
  Python row: 42
  Region: hpc
  Total spikes: 15270

Number presentation counts:
number
1    53
2    67
3    52
4    62
5    51
6    48
7    65
8    49
9    47
Name: count, dtype: int64

Firing-rate tuning:
        count      mean       std       sem
number                                     
1          53  6.498952  2.725109  0.374323
2          67  6.766169  2.706018  0.330593
3          52  7.307692  3.003638  0.416530
4          62  6.308244  2.836531  0.360240
5          51  7.189542  2.857547  0.400137
6          48  7.129630  2.720046  0.392605
7          65  6.034188  2.484281  0.308137
8          49  6.190476  2.585970  0.369424
9          47  7.092199  2.447003  0.356932


In [8]:
presentations = result["presentations"]
spike_times = result["spike_times"]

print("Presentations:", len(presentations))
print(presentations.head())
print("Number of spikes:", len(spike_times))

Presentations: 494
   trial  number          onset  operand  spike_count  firing_rate
0      1       7    6660.833333        1            6     6.666667
1      2       7   81376.233333        1            6     6.666667
2      3       7   89626.100000        1            6     6.666667
4      5       8  118708.933333        1            4     4.444444
5      6       4  124642.166667        1            3     3.333333
Number of spikes: 15270


In [9]:
X, y, edges = make_temporal_features(
    presentations,
    spike_times,
    bin_size=100,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Bin edges:", edges)

print("\nMinimum value in X:", X.min())
print("Maximum value in X:", X.max())

print("\nFirst 5 rows:")
print(X[:5])

print("\nFirst 20 labels:")
print(y[:20])

X shape: (494, 9)
y shape: (494,)
Bin edges: [ 50. 150. 250. 350. 450. 550. 650. 750. 850. 950.]

Minimum value in X: 0.0
Maximum value in X: 6.0

First 5 rows:
[[1. 0. 1. 0. 0. 0. 0. 2. 2.]
 [1. 0. 1. 1. 1. 1. 0. 1. 0.]
 [3. 0. 0. 0. 0. 1. 2. 0. 0.]
 [0. 0. 1. 0. 0. 1. 1. 0. 1.]
 [0. 1. 0. 0. 1. 0. 0. 1. 0.]]

First 20 labels:
[7 7 7 8 4 7 3 4 6 7 7 9 7 1 3 3 7 4 5 6]


In [10]:
import numpy as np

classes, counts = np.unique(y, return_counts=True)

print("\nClass counts:")
for c, n in zip(classes, counts):
    print(f"Number {c}: {n}")


Class counts:
Number 1: 53
Number 2: 67
Number 3: 52
Number 4: 62
Number 5: 51
Number 6: 48
Number 7: 65
Number 8: 49
Number 9: 47


In [11]:
poisson_result = cv_poisson_nb(
    X,
    y,
    n_splits=5,
    grouped=False,
)

print(
    "Poisson Naive Bayes accuracy:",
    poisson_result["accuracy"]
)

print(
    "Chance level:",
    1 / 9
)

Poisson Naive Bayes accuracy: 0.1214574898785425
Chance level: 0.1111111111111111


In [12]:
print("Posterior shape:", poisson_result["posterior"].shape)

print("\nFirst posterior:")
print(poisson_result["posterior"][0])

print("\nTrue number:")
print(poisson_result["y_true"][0])

print("\nPredicted number:")
print(poisson_result["y_pred"][0])

print("\nPosterior sum:")
print(poisson_result["posterior"][0].sum())

Posterior shape: (494, 9)

First posterior:
[0.04794795 0.05950325 0.1752669  0.07907041 0.02028963 0.04386829
 0.08277947 0.09650266 0.39477143]

True number:
7

Predicted number:
9

Posterior sum:
0.9999999999999993


In [13]:
groups = presentations["trial"].to_numpy()

poisson_grouped = cv_poisson_nb(
    X,
    y,
    groups=groups,
    n_splits=5,
    grouped=True,
)

print(
    "Grouped Poisson NB accuracy:",
    poisson_grouped["accuracy"]
)

print(
    "Chance:",
    1 / 9
)

Grouped Poisson NB accuracy: 0.11538461538461539
Chance: 0.1111111111111111


In [14]:
import pandas as pd
import numpy as np

posterior = poisson_grouped["posterior"]
y_true = poisson_grouped["y_true"]

posterior_matrix = np.zeros((9, 9))

for true_number in range(1, 10):
    mask = y_true == true_number

    posterior_matrix[true_number - 1] = (
        posterior[mask].mean(axis=0)
    )

posterior_df = pd.DataFrame(
    posterior_matrix,
    index=[f"True {k}" for k in range(1, 10)],
    columns=[f"P({k})" for k in range(1, 10)],
)

posterior_df

,P(1),P(2),P(3),P(4),P(5),P(6),P(7),P(8),P(9)
True 1,0.115558,0.113492,0.106159,0.105784,0.123945,0.108111,0.108232,0.106638,0.112082
True 2,0.109089,0.106271,0.110458,0.113423,0.117860,0.111535,0.111743,0.120798,0.098824
True 3,0.103527,0.112814,0.122007,0.109848,0.119479,0.104015,0.104485,0.105214,0.118611
True 4,0.102991,0.115739,0.111379,0.113889,0.106043,0.113169,0.113950,0.114904,0.107936
True 5,0.114448,0.117997,0.113327,0.106724,0.112565,0.124692,0.103760,0.103554,0.102932
True 6,0.105938,0.113313,0.099597,0.114815,0.126601,0.113712,0.103086,0.118161,0.104779
True 7,0.107950,0.109691,0.105932,0.112907,0.101233,0.106644,0.120739,0.111796,0.123107
True 8,0.109321,0.121153,0.098109,0.114098,0.106530,0.112680,0.109623,0.111308,0.117177
True 9,0.105745,0.102856,0.122807,0.109764,0.099621,0.097994,0.126249,0.113757,0.121206


In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    poisson_grouped["y_true"],
    poisson_grouped["y_pred"],
    labels=np.arange(1, 10),
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {k}" for k in range(1, 10)],
    columns=[f"Pred {k}" for k in range(1, 10)],
)

cm_df

,Pred 1,Pred 2,Pred 3,Pred 4,Pred 5,Pred 6,Pred 7,Pred 8,Pred 9
True 1,9,3,2,4,11,3,7,6,8
True 2,9,1,7,4,10,8,6,14,8
True 3,2,3,9,2,11,5,5,8,7
True 4,7,3,8,7,4,7,10,9,7
True 5,8,4,5,3,5,10,3,6,7
True 6,6,0,3,7,10,5,3,7,7
True 7,8,1,10,11,3,7,8,6,11
True 8,6,2,2,6,5,5,5,6,12
True 9,3,1,4,6,3,4,13,6,7


In [16]:
from src.poisson_naive_bayes import fit_poisson_nb

model_full = fit_poisson_nb(
    X,
    y,
    uniform_prior=True,
)

print(model_full["rates"].shape)
print(model_full["rates"])

(9, 9)
[[0.67924528 0.52830189 0.88679245 0.58490566 0.62264151 0.45283019
  0.60377358 0.67924528 0.81132075]
 [0.70149254 0.71641791 0.71641791 0.7761194  0.68656716 0.70149254
  0.58208955 0.6119403  0.59701493]
 [0.78846154 0.94230769 0.51923077 0.63461538 0.69230769 0.69230769
  0.78846154 0.69230769 0.82692308]
 [0.5483871  0.72580645 0.61290323 0.61290323 0.74193548 0.70967742
  0.62903226 0.61290323 0.48387097]
 [0.49019608 0.7254902  0.84313725 0.82352941 0.62745098 0.68627451
  0.74509804 0.78431373 0.74509804]
 [0.60416667 0.64583333 0.66666667 0.875      0.79166667 0.77083333
  0.5        0.875      0.6875    ]
 [0.6        0.46153846 0.41538462 0.64615385 0.69230769 0.50769231
  0.72307692 0.78461538 0.6       ]
 [0.79591837 0.59183673 0.69387755 0.69387755 0.51020408 0.57142857
  0.51020408 0.75510204 0.44897959]
 [0.93617021 0.70212766 0.61702128 0.55319149 0.87234043 0.42553191
  0.93617021 0.87234043 0.46808511]]


In [17]:
import pandas as pd

lambda_df = pd.DataFrame(
    model_full["rates"],
    index=[f"Number {k}" for k in range(1, 10)],
    columns=[
        "50-150",
        "150-250",
        "250-350",
        "350-450",
        "450-550",
        "550-650",
        "650-750",
        "750-850",
        "850-950",
    ],
)

lambda_df

,50-150,150-250,250-350,350-450,450-550,550-650,650-750,750-850,850-950
Number 1,0.679245,0.528302,0.886792,0.584906,0.622642,0.452830,0.603774,0.679245,0.811321
Number 2,0.701493,0.716418,0.716418,0.776119,0.686567,0.701493,0.582090,0.611940,0.597015
Number 3,0.788462,0.942308,0.519231,0.634615,0.692308,0.692308,0.788462,0.692308,0.826923
Number 4,0.548387,0.725806,0.612903,0.612903,0.741935,0.709677,0.629032,0.612903,0.483871
Number 5,0.490196,0.725490,0.843137,0.823529,0.627451,0.686275,0.745098,0.784314,0.745098
Number 6,0.604167,0.645833,0.666667,0.875000,0.791667,0.770833,0.500000,0.875000,0.687500
Number 7,0.600000,0.461538,0.415385,0.646154,0.692308,0.507692,0.723077,0.784615,0.600000
Number 8,0.795918,0.591837,0.693878,0.693878,0.510204,0.571429,0.510204,0.755102,0.448980
Number 9,0.936170,0.702128,0.617021,0.553191,0.872340,0.425532,0.936170,0.872340,0.468085


In [18]:
import numpy as np
import pandas as pd

classes = np.arange(1, 10)

fano_matrix = np.full((9, 9), np.nan)
mean_matrix = np.full((9, 9), np.nan)
variance_matrix = np.full((9, 9), np.nan)

for i, k in enumerate(classes):

    X_k = X[y == k]

    # Mean spike count in each 100-ms bin
    means = X_k.mean(axis=0)

    # Sample variance in each bin
    variances = X_k.var(axis=0, ddof=1)

    mean_matrix[i] = means
    variance_matrix[i] = variances

    # Avoid division by zero
    fano = np.divide(
        variances,
        means,
        out=np.full_like(means, np.nan, dtype=float),
        where=means > 0,
    )

    fano_matrix[i] = fano

In [19]:
bin_labels = [
    "50-150",
    "150-250",
    "250-350",
    "350-450",
    "450-550",
    "550-650",
    "650-750",
    "750-850",
    "850-950",
]

fano_df = pd.DataFrame(
    fano_matrix,
    index=[f"Number {k}" for k in classes],
    columns=bin_labels,
)

fano_df

,50-150,150-250,250-350,350-450,450-550,550-650,650-750,750-850,850-950
Number 1,1.459402,0.844780,1.026187,1.212159,1.187646,0.982372,0.786058,1.176282,0.903399
Number 2,0.691812,0.626263,0.837753,0.773893,1.200922,0.994197,0.944833,1.037694,0.865909
Number 3,0.862267,0.932773,1.018882,0.866904,1.333333,0.823529,1.508847,0.993464,0.935249
Number 4,1.296046,1.136976,0.928387,0.714409,1.101924,1.126677,1.158890,0.660915,0.931148
Number 5,0.846400,0.996757,1.108837,1.054286,0.762500,0.844571,0.957895,1.189000,0.904211
Number 6,1.038151,0.822924,0.659574,0.954407,1.126540,0.896492,1.191489,0.905775,0.938104
Number 7,1.135417,1.088542,0.969907,0.697917,0.944444,0.869318,1.145612,1.294118,1.343750
Number 8,1.045940,0.909483,0.912990,1.153186,0.826667,1.020833,0.826667,0.856982,0.933712
Number 9,1.272727,1.109354,0.814093,1.085284,0.728526,0.791304,0.761858,0.927890,0.822134


In [20]:
print("Mean Fano factor:", np.nanmean(fano_matrix))
print("Median Fano factor:", np.nanmedian(fano_matrix))
print("Minimum Fano factor:", np.nanmin(fano_matrix))
print("Maximum Fano factor:", np.nanmax(fano_matrix))

Mean Fano factor: 0.9798610936465963
Median Fano factor: 0.9444444444444439
Minimum Fano factor: 0.6262626262626261
Maximum Fano factor: 1.5088474414155906


In [21]:
valid_fano = fano_matrix[np.isfinite(fano_matrix)]

print(
    "Fraction between 0.8 and 1.2:",
    np.mean((valid_fano >= 0.8) & (valid_fano <= 1.2))
)

print(
    "Fraction between 0.5 and 1.5:",
    np.mean((valid_fano >= 0.5) & (valid_fano <= 1.5))
)

Fraction between 0.8 and 1.2: 0.7407407407407407
Fraction between 0.5 and 1.5: 0.9876543209876543


-----subject level 

In [22]:
from src.poisson_subject_analysis import (
    analyze_poisson_subject,
)

YFU_poisson = analyze_poisson_subject(
    "YFU",
    root=PROJECT_ROOT,
    bin_size=100,
    n_splits=5,
    mtl_only=True,
)

YFU: neuron 1/78
YFU: neuron 2/78
YFU: neuron 3/78
YFU: neuron 4/78
YFU: neuron 5/78
YFU: neuron 6/78
YFU: neuron 7/78
YFU: neuron 8/78
YFU: neuron 9/78
YFU: neuron 10/78
YFU: neuron 11/78
YFU: neuron 12/78
YFU: neuron 13/78
YFU: neuron 14/78
YFU: neuron 15/78
YFU: neuron 16/78
YFU: neuron 17/78
YFU: neuron 18/78
YFU: neuron 19/78
YFU: neuron 20/78
YFU: neuron 21/78
YFU: neuron 22/78
YFU: neuron 23/78
YFU: neuron 24/78
YFU: neuron 25/78
YFU: neuron 26/78
YFU: neuron 27/78
YFU: neuron 28/78
YFU: neuron 29/78
YFU: neuron 30/78
YFU: neuron 31/78
YFU: neuron 32/78
YFU: neuron 33/78
YFU: neuron 34/78
YFU: neuron 35/78
YFU: neuron 36/78
YFU: neuron 37/78
YFU: neuron 38/78
YFU: neuron 39/78
YFU: neuron 40/78
YFU: neuron 41/78
YFU: neuron 42/78
YFU: neuron 43/78
YFU: neuron 44/78
YFU: neuron 45/78
YFU: neuron 46/78
YFU: neuron 47/78
YFU: neuron 48/78
YFU: neuron 49/78
YFU: neuron 50/78
YFU: neuron 51/78
YFU: neuron 52/78
YFU: neuron 53/78
YFU: neuron 54/78
YFU: neuron 55/78
YFU: neuron 56/78
Y

In [23]:
YFU_summary = YFU_poisson["summary"]

YFU_summary

,subject,neuron,region,n_presentations,bin_size_ms,accuracy,chance,mean_fano,median_fano,frac_fano_0.8_1.2,frac_fano_0.5_1.5
0,YFU,1,amy,494,100,0.107287,0.111111,1.234002,0.992872,0.506173,0.864198
1,YFU,2,amy,494,100,0.107287,0.111111,0.980429,0.958984,0.777778,1.000000
2,YFU,3,amy,494,100,0.129555,0.111111,1.012548,1.020833,0.765432,1.000000
3,YFU,4,amy,494,100,0.103239,0.111111,1.018563,1.016872,0.691358,1.000000
4,YFU,5,amy,494,100,0.117409,0.111111,1.003196,1.000000,0.975000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
57,YFU,66,hpc,494,100,0.119433,0.111111,1.258049,1.217772,0.419753,0.827160
58,YFU,67,hpc,494,100,0.101215,0.111111,0.987109,0.968427,0.765432,1.000000
59,YFU,68,hpc,494,100,0.137652,0.111111,1.087400,1.061215,0.728395,0.950617
60,YFU,69,hpc,494,100,0.113360,0.111111,1.109552,1.053846,0.641975,0.962963


In [24]:
print(
    YFU_summary[
        [
            "accuracy",
            "mean_fano",
            "median_fano",
        ]
    ].describe()
)

        accuracy  mean_fano  median_fano
count  62.000000  62.000000    62.000000
mean    0.111205   1.007150     0.977715
std     0.017039   0.069403     0.059972
min     0.076923   0.785252     0.771400
25%     0.101215   0.976719     0.944878
50%     0.112348   0.995116     0.970430
75%     0.121457   1.031672     1.012254
max     0.155870   1.258049     1.217772


In [25]:
YFU_summary.sort_values(
    "accuracy",
    ascending=False,
).head(10)

,subject,neuron,region,n_presentations,bin_size_ms,accuracy,chance,mean_fano,median_fano,frac_fano_0.8_1.2,frac_fano_0.5_1.5
25,YFU,34,hpc,494,100,0.155870,0.111111,0.964538,0.941622,0.728395,1.000000
43,YFU,52,amy,494,100,0.145749,0.111111,0.957255,0.942308,0.975000,1.000000
59,YFU,68,hpc,494,100,0.137652,0.111111,1.087400,1.061215,0.728395,0.950617
53,YFU,62,hpc,494,100,0.133603,0.111111,1.009228,0.916667,0.753086,0.962963
30,YFU,39,hpc,494,100,0.133603,0.111111,0.944995,0.917922,0.679012,1.000000
31,YFU,40,hpc,494,100,0.131579,0.111111,1.042926,0.978723,0.883117,0.974026
2,YFU,3,amy,494,100,0.129555,0.111111,1.012548,1.020833,0.765432,1.000000
5,YFU,6,amy,494,100,0.129555,0.111111,1.015191,0.999132,0.740741,0.987654
41,YFU,50,amy,494,100,0.129555,0.111111,1.014091,1.016667,0.765432,1.000000
40,YFU,49,amy,494,100,0.127530,0.111111,0.996235,0.946181,0.567901,0.962963


In [26]:
YFU_poisson["confusion"][43]

array([[ 9,  3,  2,  4, 11,  3,  7,  6,  8],
       [ 9,  1,  7,  4, 10,  8,  6, 14,  8],
       [ 2,  3,  9,  2, 11,  5,  5,  8,  7],
       [ 7,  3,  8,  7,  4,  7, 10,  9,  7],
       [ 8,  4,  5,  3,  5, 10,  3,  6,  7],
       [ 6,  0,  3,  7, 10,  5,  3,  7,  7],
       [ 8,  1, 10, 11,  3,  7,  8,  6, 11],
       [ 6,  2,  2,  6,  5,  5,  5,  6, 12],
       [ 3,  1,  4,  6,  3,  4, 13,  6,  7]])

In [27]:
YFU_summary.sort_values(
    "accuracy",
    ascending=False
)[
    [
        "neuron",
        "region",
        "accuracy",
        "mean_fano",
        "median_fano",
    ]
].head(10)

,neuron,region,accuracy,mean_fano,median_fano
25,34,hpc,0.155870,0.964538,0.941622
43,52,amy,0.145749,0.957255,0.942308
59,68,hpc,0.137652,1.087400,1.061215
53,62,hpc,0.133603,1.009228,0.916667
30,39,hpc,0.133603,0.944995,0.917922
31,40,hpc,0.131579,1.042926,0.978723
2,3,amy,0.129555,1.012548,1.020833
5,6,amy,0.129555,1.015191,0.999132
41,50,amy,0.129555,1.014091,1.016667
40,49,amy,0.127530,0.996235,0.946181


In [28]:
import numpy as np
import pandas as pd

MTL_REGIONS = {"hpc", "amy", "ent", "phc"}

population_columns = []
population_neurons = []

reference_presentations = None

# YFU has 78 recorded neurons total
for neuron in range(1, 79):

    result = analyze_neuron(
        "YFU",
        neuron,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    region = result["region"]

    if region not in MTL_REGIONS:
        continue

    presentations_n = result["presentations"].reset_index(drop=True)

    # -------------------------------------------------
    # Make sure every neuron has exactly the same
    # behavioral presentations in exactly the same order
    # -------------------------------------------------

    if reference_presentations is None:

        reference_presentations = presentations_n[
            ["trial", "number", "onset", "operand"]
        ].copy()

    else:

        assert np.array_equal(
            presentations_n["trial"].to_numpy(),
            reference_presentations["trial"].to_numpy(),
        )

        assert np.array_equal(
            presentations_n["number"].to_numpy(),
            reference_presentations["number"].to_numpy(),
        )

        assert np.allclose(
            presentations_n["onset"].to_numpy(),
            reference_presentations["onset"].to_numpy(),
        )

    # One 900-ms spike count per presentation
    population_columns.append(
        presentations_n["spike_count"].to_numpy(dtype=float)
    )

    population_neurons.append(
        {
            "neuron": neuron,
            "region": region,
        }
    )

In [29]:
X_pop = np.column_stack(population_columns)

y_pop = reference_presentations[
    "number"
].to_numpy()

groups_pop = reference_presentations[
    "trial"
].to_numpy()

population_neurons_df = pd.DataFrame(
    population_neurons
)

print("Population X shape:", X_pop.shape)
print("y shape:", y_pop.shape)

print("\nMinimum spike count:", X_pop.min())
print("Maximum spike count:", X_pop.max())

print("\nNeurons included:")
print(population_neurons_df)

Population X shape: (494, 62)
y shape: (494,)

Minimum spike count: 0.0
Maximum spike count: 61.0

Neurons included:
    neuron region
0        1    amy
1        2    amy
2        3    amy
3        4    amy
4        5    amy
..     ...    ...
57      66    hpc
58      67    hpc
59      68    hpc
60      69    hpc
61      70    hpc

[62 rows x 2 columns]


In [30]:
classes = np.arange(1, 10)

lambda_pop = np.full(
    (9, X_pop.shape[1]),
    np.nan,
)

fano_pop = np.full(
    (9, X_pop.shape[1]),
    np.nan,
)

for i, k in enumerate(classes):

    X_k = X_pop[y_pop == k]

    means = X_k.mean(axis=0)

    variances = X_k.var(
        axis=0,
        ddof=1,
    )

    lambda_pop[i] = means

    fano_pop[i] = np.divide(
        variances,
        means,
        out=np.full_like(
            means,
            np.nan,
            dtype=float,
        ),
        where=means > 0,
    )

valid_fano_pop = fano_pop[
    np.isfinite(fano_pop)
]

print("Mean Fano:", np.mean(valid_fano_pop))
print("Median Fano:", np.median(valid_fano_pop))
print("Minimum Fano:", np.min(valid_fano_pop))
print("Maximum Fano:", np.max(valid_fano_pop))

print(
    "Fraction 0.8-1.2:",
    np.mean(
        (valid_fano_pop >= 0.8)
        &
        (valid_fano_pop <= 1.2)
    )
)

print(
    "Fraction 0.5-1.5:",
    np.mean(
        (valid_fano_pop >= 0.5)
        &
        (valid_fano_pop <= 1.5)
    )
)

Mean Fano: 1.1795944742935203
Median Fano: 1.0774724230906143
Minimum Fano: 0.4311926605504588
Maximum Fano: 6.089822869720152
Fraction 0.8-1.2: 0.5663082437275986
Fraction 0.5-1.5: 0.8888888888888888


In [31]:
population_result = cv_poisson_nb(
    X_pop,
    y_pop,
    groups=groups_pop,
    n_splits=5,
    grouped=True,
)

print(
    "Population Poisson NB accuracy:",
    population_result["accuracy"]
)

print(
    "Chance:",
    1 / 9
)

Population Poisson NB accuracy: 0.08906882591093117
Chance: 0.1111111111111111


In [32]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

cm_pop = confusion_matrix(
    population_result["y_true"],
    population_result["y_pred"],
    labels=np.arange(1, 10),
)

cm_pop_df = pd.DataFrame(
    cm_pop,
    index=np.arange(1, 10),
    columns=np.arange(1, 10),
)

cm_pop_df

,1,2,3,4,5,6,7,8,9
1,3,5,4,5,6,12,7,6,5
2,6,4,9,12,7,8,8,7,6
3,7,1,3,8,8,9,2,7,7
4,5,6,6,11,7,4,7,11,5
5,6,6,3,10,4,4,6,3,9
6,10,3,7,5,4,4,7,3,5
7,3,7,5,11,11,6,9,10,3
8,7,1,6,8,9,5,7,1,5
9,7,5,4,8,7,1,5,5,5


In [33]:
predicted_counts = pd.Series(
    population_result["y_pred"]
).value_counts().sort_index()

predicted_counts

1    54
2    38
3    47
4    78
5    63
6    53
7    58
8    53
9    50
Name: count, dtype: int64

In [34]:
import numpy as np
import pandas as pd

MTL_REGIONS = {"hpc", "amy", "ent", "phc"}

temporal_blocks = []
temporal_neurons = []

reference_presentations = None
bin_edges = None

for neuron in range(1, 79):

    result = analyze_neuron(
        "YFU",
        neuron,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    region = result["region"]

    if region not in MTL_REGIONS:
        continue

    presentations_n = (
        result["presentations"]
        .reset_index(drop=True)
    )

    spike_times = result["spike_times"]

    # ------------------------------------
    # Check presentation alignment
    # ------------------------------------

    if reference_presentations is None:

        reference_presentations = presentations_n[
            ["trial", "number", "onset", "operand"]
        ].copy()

    else:

        assert np.array_equal(
            presentations_n["trial"].to_numpy(),
            reference_presentations["trial"].to_numpy(),
        )

        assert np.array_equal(
            presentations_n["number"].to_numpy(),
            reference_presentations["number"].to_numpy(),
        )

        assert np.allclose(
            presentations_n["onset"].to_numpy(),
            reference_presentations["onset"].to_numpy(),
        )

    # ------------------------------------
    # 100-ms temporal counts
    # ------------------------------------

    X_neuron, y_neuron, edges = make_temporal_features(
        presentations_n,
        spike_times,
        bin_size=100,
        window_start=50,
        window_end=950,
    )

    temporal_blocks.append(X_neuron)

    temporal_neurons.append(
        {
            "neuron": neuron,
            "region": region,
        }
    )

    bin_edges = edges

In [35]:
X_temporal_pop = np.concatenate(
    temporal_blocks,
    axis=1,
)

y_temporal_pop = (
    reference_presentations["number"]
    .to_numpy()
)

groups_temporal_pop = (
    reference_presentations["trial"]
    .to_numpy()
)

temporal_neurons_df = pd.DataFrame(
    temporal_neurons
)

print(
    "Temporal population shape:",
    X_temporal_pop.shape
)

print(
    "Labels shape:",
    y_temporal_pop.shape
)

print(
    "Minimum count:",
    X_temporal_pop.min()
)

print(
    "Maximum count:",
    X_temporal_pop.max()
)

print(
    "Bin edges:",
    bin_edges
)

Temporal population shape: (494, 558)
Labels shape: (494,)
Minimum count: 0.0
Maximum count: 12.0
Bin edges: [ 50. 150. 250. 350. 450. 550. 650. 750. 850. 950.]


In [36]:
feature_names = []

for _, row in temporal_neurons_df.iterrows():

    neuron = row["neuron"]
    region = row["region"]

    for b in range(9):

        start = int(bin_edges[b])
        end = int(bin_edges[b + 1])

        feature_names.append(
            f"{region}.{neuron}_{start}-{end}ms"
        )

print(len(feature_names))
print(feature_names[:20])

558
['amy.1_50-150ms', 'amy.1_150-250ms', 'amy.1_250-350ms', 'amy.1_350-450ms', 'amy.1_450-550ms', 'amy.1_550-650ms', 'amy.1_650-750ms', 'amy.1_750-850ms', 'amy.1_850-950ms', 'amy.2_50-150ms', 'amy.2_150-250ms', 'amy.2_250-350ms', 'amy.2_350-450ms', 'amy.2_450-550ms', 'amy.2_550-650ms', 'amy.2_650-750ms', 'amy.2_750-850ms', 'amy.2_850-950ms', 'amy.3_50-150ms', 'amy.3_150-250ms']


In [37]:
first_presentation = pd.Series(
    X_temporal_pop[0],
    index=feature_names,
)

first_presentation.head(20)

amy.1_50-150ms     1.0
amy.1_150-250ms    0.0
amy.1_250-350ms    0.0
amy.1_350-450ms    0.0
amy.1_450-550ms    0.0
amy.1_550-650ms    0.0
amy.1_650-750ms    0.0
amy.1_750-850ms    0.0
amy.1_850-950ms    0.0
amy.2_50-150ms     0.0
amy.2_150-250ms    2.0
amy.2_250-350ms    1.0
amy.2_350-450ms    1.0
amy.2_450-550ms    2.0
amy.2_550-650ms    1.0
amy.2_650-750ms    2.0
amy.2_750-850ms    2.0
amy.2_850-950ms    2.0
amy.3_50-150ms     0.0
amy.3_150-250ms    1.0
dtype: float64

In [38]:
y_temporal_pop[0]

np.int64(7)

In [39]:
classes = np.arange(1, 10)

lambda_temporal_pop = np.full(
    (9, X_temporal_pop.shape[1]),
    np.nan,
)

fano_temporal_pop = np.full(
    (9, X_temporal_pop.shape[1]),
    np.nan,
)

for i, k in enumerate(classes):

    # All presentations where number k was shown
    X_k = X_temporal_pop[
        y_temporal_pop == k
    ]

    # Mean count = Poisson lambda estimate
    means = X_k.mean(axis=0)

    # Across-presentation variance
    variances = X_k.var(
        axis=0,
        ddof=1,
    )

    lambda_temporal_pop[i] = means

    fano_temporal_pop[i] = np.divide(
        variances,
        means,
        out=np.full_like(
            means,
            np.nan,
            dtype=float,
        ),
        where=means > 0,
    )

In [40]:
print(
    "Lambda matrix shape:",
    lambda_temporal_pop.shape
)

print(
    "Fano matrix shape:",
    fano_temporal_pop.shape
)

Lambda matrix shape: (9, 558)
Fano matrix shape: (9, 558)


In [41]:
valid_fano_temporal = (
    fano_temporal_pop[
        np.isfinite(fano_temporal_pop)
    ]
)

print(
    "Mean Fano:",
    np.mean(valid_fano_temporal)
)

print(
    "Median Fano:",
    np.median(valid_fano_temporal)
)

print(
    "Minimum Fano:",
    np.min(valid_fano_temporal)
)

print(
    "Maximum Fano:",
    np.max(valid_fano_temporal)
)

print(
    "Fraction 0.8-1.2:",
    np.mean(
        (valid_fano_temporal >= 0.8)
        &
        (valid_fano_temporal <= 1.2)
    )
)

print(
    "Fraction 0.5-1.5:",
    np.mean(
        (valid_fano_temporal >= 0.5)
        &
        (valid_fano_temporal <= 1.5)
    )
)

Mean Fano: 1.0070724513417961
Median Fano: 0.9776162899346543
Minimum Fano: 0.4822695035460993
Maximum Fano: 5.750759878419451
Fraction 0.8-1.2: 0.7263349514563107
Fraction 0.5-1.5: 0.9747168284789643


In [42]:
fano_reshaped = fano_temporal_pop.reshape(
    9,   # numbers
    62,  # neurons
    9,   # temporal bins
)

mean_fano_by_bin = np.nanmean(
    fano_reshaped,
    axis=(0, 1),
)

median_fano_by_bin = np.nanmedian(
    fano_reshaped,
    axis=(0, 1),
)

fano_by_bin_df = pd.DataFrame(
    {
        "bin_start_ms": bin_edges[:-1],
        "bin_end_ms": bin_edges[1:],
        "mean_fano": mean_fano_by_bin,
        "median_fano": median_fano_by_bin,
    }
)

fano_by_bin_df

,bin_start_ms,bin_end_ms,mean_fano,median_fano
0,50.0,150.0,1.019332,0.979885
1,150.0,250.0,1.010013,0.980474
2,250.0,350.0,0.989151,0.968116
3,350.0,450.0,0.999893,0.958333
4,450.0,550.0,1.007651,0.984848
5,550.0,650.0,1.002440,0.980769
6,650.0,750.0,1.012067,0.972136
7,750.0,850.0,1.010223,0.978723
8,850.0,950.0,1.012926,0.980000


In [43]:
temporal_population_result = cv_poisson_nb(
    X_temporal_pop,
    y_temporal_pop,
    groups=groups_temporal_pop,
    n_splits=5,
    grouped=True,
)

print(
    "Temporal population accuracy:",
    temporal_population_result["accuracy"]
)

print(
    "Chance:",
    1 / 9
)

Temporal population accuracy: 0.10931174089068826
Chance: 0.1111111111111111


In [44]:
time_bin_results = []

for b in range(9):

    # Pick the same temporal bin from every neuron
    cols = []

    for neuron_idx in range(62):
        col = neuron_idx * 9 + b
        cols.append(col)

    X_bin = X_temporal_pop[:, cols]

    result_bin = cv_poisson_nb(
        X_bin,
        y_temporal_pop,
        groups=groups_temporal_pop,
        n_splits=5,
        grouped=True,
    )

    accuracy = result_bin["accuracy"]

    time_bin_results.append(
        {
            "bin": b + 1,
            "start_ms": bin_edges[b],
            "end_ms": bin_edges[b + 1],
            "accuracy": accuracy,
        }
    )

In [46]:
time_bin_df = pd.DataFrame(
    time_bin_results
)

time_bin_df

,bin,start_ms,end_ms,accuracy
0,1,50.0,150.0,0.113360
1,2,150.0,250.0,0.082996
2,3,250.0,350.0,0.129555
3,4,350.0,450.0,0.121457
4,5,450.0,550.0,0.109312
5,6,550.0,650.0,0.109312
6,7,650.0,750.0,0.099190
7,8,750.0,850.0,0.131579
8,9,850.0,950.0,0.095142


In [47]:
time_bin_df["chance"] = 1 / 9

time_bin_df

,bin,start_ms,end_ms,accuracy,chance
0,1,50.0,150.0,0.113360,0.111111
1,2,150.0,250.0,0.082996,0.111111
2,3,250.0,350.0,0.129555,0.111111
3,4,350.0,450.0,0.121457,0.111111
4,5,450.0,550.0,0.109312,0.111111
5,6,550.0,650.0,0.109312,0.111111
6,7,650.0,750.0,0.099190,0.111111
7,8,750.0,850.0,0.131579,0.111111
8,9,850.0,950.0,0.095142,0.111111


In [48]:
best_bin_row = time_bin_df.loc[
    time_bin_df["accuracy"].idxmax()
]

best_bin_row

bin           8.000000
start_ms    750.000000
end_ms      850.000000
accuracy      0.131579
chance        0.111111
Name: 7, dtype: float64

In [49]:
print(
    "Mean bin-wise accuracy:",
    time_bin_df["accuracy"].mean()
)

print(
    "Maximum bin accuracy:",
    time_bin_df["accuracy"].max()
)

print(
    "Chance:",
    1 / 9
)

Mean bin-wise accuracy: 0.11021142600089967
Maximum bin accuracy: 0.13157894736842105
Chance: 0.1111111111111111


In [50]:
mtl_neuron_cache = []

for neuron in population_neurons_df["neuron"]:

    result = analyze_neuron(
        "YFU",
        int(neuron),
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    mtl_neuron_cache.append(
        {
            "neuron": int(neuron),
            "region": result["region"],
            "presentations":
                result["presentations"].reset_index(drop=True),
            "spike_times":
                result["spike_times"],
        }
    )

print(
    "Cached MTL neurons:",
    len(mtl_neuron_cache)
)


Cached MTL neurons: 62


In [52]:
def compute_fano_lambda(X, y):
    classes = np.arange(1, 10)

    n_features = X.shape[1]

    lambda_matrix = np.full(
        (9, n_features),
        np.nan,
    )

    fano_matrix = np.full(
        (9, n_features),
        np.nan,
    )

    for i, k in enumerate(classes):

        X_k = X[y == k]

        means = X_k.mean(axis=0)

        variances = X_k.var(
            axis=0,
            ddof=1,
        )

        lambda_matrix[i] = means

        fano_matrix[i] = np.divide(
            variances,
            means,
            out=np.full(
                n_features,
                np.nan,
                dtype=float,
            ),
            where=means > 0,
        )

    return lambda_matrix, fano_matrix

In [53]:
from sklearn.metrics import balanced_accuracy_score

bin_sizes = [
    30,
    60,
    90,
    100,
    150,
    180,
    225,
    300,
    450,
    900,
]

bin_size_results = []

for bin_size in bin_sizes:

    print(f"Running bin size = {bin_size} ms")

    blocks = []

    for neuron_data in mtl_neuron_cache:

        X_n, y_n, edges = make_temporal_features(
            neuron_data["presentations"],
            neuron_data["spike_times"],
            bin_size=bin_size,
            window_start=50,
            window_end=950,
        )

        blocks.append(X_n)

    # -----------------------------------
    # Population matrix
    # -----------------------------------

    X_current = np.concatenate(
        blocks,
        axis=1,
    )

    y_current = y_n

    groups_current = (
        mtl_neuron_cache[0]
        ["presentations"]["trial"]
        .to_numpy()
    )

    # -----------------------------------
    # Fano factors
    # -----------------------------------

    _, fano_current = compute_fano_lambda(
        X_current,
        y_current,
    )

    valid_fano = fano_current[
        np.isfinite(fano_current)
    ]

    # -----------------------------------
    # Grouped CV
    # -----------------------------------

    cv_result = cv_poisson_nb(
        X_current,
        y_current,
        groups=groups_current,
        n_splits=5,
        grouped=True,
        random_state=42,
    )

    balanced_acc = balanced_accuracy_score(
        cv_result["y_true"],
        cv_result["y_pred"],
    )

    bin_size_results.append(
        {
            "bin_size_ms": bin_size,
            "n_bins": X_current.shape[1] // 62,
            "n_features": X_current.shape[1],
            "accuracy": cv_result["accuracy"],
            "balanced_accuracy": balanced_acc,
            "mean_fano": np.mean(valid_fano),
            "median_fano": np.median(valid_fano),
            "frac_fano_0.8_1.2":
                np.mean(
                    (valid_fano >= 0.8)
                    &
                    (valid_fano <= 1.2)
                ),
        }
    )

Running bin size = 30 ms
Running bin size = 60 ms
Running bin size = 90 ms
Running bin size = 100 ms
Running bin size = 150 ms
Running bin size = 180 ms
Running bin size = 225 ms
Running bin size = 300 ms
Running bin size = 450 ms
Running bin size = 900 ms


In [55]:
bin_size_df = pd.DataFrame(
    bin_size_results
)

bin_size_df

,bin_size_ms,n_bins,n_features,accuracy,balanced_accuracy,mean_fano,median_fano,frac_fano_0.8_1.2
0,30,30,1860,0.129555,0.119976,0.984662,0.956522,0.806792
1,60,15,930,0.099190,0.095334,0.997441,0.969954,0.746804
2,90,10,620,0.111336,0.109861,1.007752,0.979167,0.728582
3,100,9,558,0.109312,0.105901,1.007072,0.977616,0.726335
4,150,6,372,0.089069,0.088494,1.027457,0.984472,0.697079
5,180,5,310,0.101215,0.100499,1.038774,0.992232,0.691060
6,225,4,248,0.080972,0.078408,1.055035,1.003397,0.674607
7,300,3,186,0.076923,0.074810,1.072225,1.016813,0.657485
8,450,2,124,0.087045,0.084533,1.105580,1.041291,0.622980
9,900,1,62,0.091093,0.088671,1.179603,1.076409,0.568100


In [56]:
bin_size_df.sort_values(
    "accuracy",
    ascending=False,
)

,bin_size_ms,n_bins,n_features,accuracy,balanced_accuracy,mean_fano,median_fano,frac_fano_0.8_1.2
0,30,30,1860,0.129555,0.119976,0.984662,0.956522,0.806792
2,90,10,620,0.111336,0.109861,1.007752,0.979167,0.728582
3,100,9,558,0.109312,0.105901,1.007072,0.977616,0.726335
5,180,5,310,0.101215,0.100499,1.038774,0.992232,0.691060
1,60,15,930,0.099190,0.095334,0.997441,0.969954,0.746804
9,900,1,62,0.091093,0.088671,1.179603,1.076409,0.568100
4,150,6,372,0.089069,0.088494,1.027457,0.984472,0.697079
8,450,2,124,0.087045,0.084533,1.105580,1.041291,0.622980
6,225,4,248,0.080972,0.078408,1.055035,1.003397,0.674607
7,300,3,186,0.076923,0.074810,1.072225,1.016813,0.657485


In [57]:
bin_size_df[
    [
        "bin_size_ms",
        "n_bins",
        "n_features",
        "accuracy",
        "balanced_accuracy",
        "mean_fano",
        "median_fano",
        "frac_fano_0.8_1.2",
    ]
]

,bin_size_ms,n_bins,n_features,accuracy,balanced_accuracy,mean_fano,median_fano,frac_fano_0.8_1.2
0,30,30,1860,0.129555,0.119976,0.984662,0.956522,0.806792
1,60,15,930,0.099190,0.095334,0.997441,0.969954,0.746804
2,90,10,620,0.111336,0.109861,1.007752,0.979167,0.728582
3,100,9,558,0.109312,0.105901,1.007072,0.977616,0.726335
4,150,6,372,0.089069,0.088494,1.027457,0.984472,0.697079
5,180,5,310,0.101215,0.100499,1.038774,0.992232,0.691060
6,225,4,248,0.080972,0.078408,1.055035,1.003397,0.674607
7,300,3,186,0.076923,0.074810,1.072225,1.016813,0.657485
8,450,2,124,0.087045,0.084533,1.105580,1.041291,0.622980
9,900,1,62,0.091093,0.088671,1.179603,1.076409,0.568100


In [58]:
X_by_bin_size = {}

for bin_size in bin_sizes:

    print(f"Building {bin_size} ms")

    blocks = []
    y_reference = None

    for neuron_data in mtl_neuron_cache:

        X_n, y_n, edges = make_temporal_features(
            neuron_data["presentations"],
            neuron_data["spike_times"],
            bin_size=bin_size,
            window_start=50,
            window_end=950,
        )

        if y_reference is None:
            y_reference = y_n.copy()
        else:
            assert np.array_equal(
                y_n,
                y_reference,
            )

        blocks.append(X_n)

    X_by_bin_size[bin_size] = np.concatenate(
        blocks,
        axis=1,
    )

y_perm_base = y_reference.copy()

groups_perm = (
    mtl_neuron_cache[0]["presentations"]["trial"]
    .to_numpy()
)

operand_perm = (
    mtl_neuron_cache[0]["presentations"]["operand"]
    .to_numpy()
)

print("Saved matrices:", list(X_by_bin_size.keys()))

Building 30 ms
Building 60 ms
Building 90 ms
Building 100 ms
Building 150 ms
Building 180 ms
Building 225 ms
Building 300 ms
Building 450 ms
Building 900 ms
Saved matrices: [30, 60, 90, 100, 150, 180, 225, 300, 450, 900]


In [59]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fixed_folds = list(
    splitter.split(
        X_by_bin_size[100],
        y_perm_base,
        groups=groups_perm,
    )
)

print("Number of folds:", len(fixed_folds))

Number of folds: 5


In [60]:
from src.poisson_naive_bayes import (
    fit_poisson_nb,
    predict_poisson_nb,
)

def fixed_fold_poisson_accuracy(X, y, folds):

    predictions = np.empty(
        len(y),
        dtype=int,
    )

    for train_idx, test_idx in folds:

        model = fit_poisson_nb(
            X[train_idx],
            y[train_idx],
            uniform_prior=True,
        )

        predictions[test_idx] = predict_poisson_nb(
            model,
            X[test_idx],
        )

    accuracy = np.mean(
        predictions == y
    )

    return accuracy

In [61]:
observed_accuracies = {}

for bin_size in bin_sizes:

    observed_accuracies[bin_size] = (
        fixed_fold_poisson_accuracy(
            X_by_bin_size[bin_size],
            y_perm_base,
            fixed_folds,
        )
    )

observed_df = pd.DataFrame(
    {
        "bin_size_ms": bin_sizes,
        "accuracy": [
            observed_accuracies[b]
            for b in bin_sizes
        ],
    }
)

observed_df

,bin_size_ms,accuracy
0,30,0.129555
1,60,0.099190
2,90,0.111336
3,100,0.109312
4,150,0.089069
5,180,0.101215
6,225,0.080972
7,300,0.076923
8,450,0.087045
9,900,0.091093


In [62]:
observed_best_bin = max(
    observed_accuracies,
    key=observed_accuracies.get,
)

observed_best_accuracy = (
    observed_accuracies[
        observed_best_bin
    ]
)

print(
    "Observed best bin:",
    observed_best_bin,
)

print(
    "Observed best accuracy:",
    observed_best_accuracy,
)

Observed best bin: 30
Observed best accuracy: 0.12955465587044535


In [63]:
rng = np.random.default_rng(42)

n_permutations = 200

null_accuracy_matrix = np.zeros(
    (
        n_permutations,
        len(bin_sizes),
    )
)

null_max_accuracy = np.zeros(
    n_permutations
)

for p in range(n_permutations):

    # ---------------------------------
    # Shuffle number labels
    # within operand position
    # ---------------------------------

    y_shuffled = y_perm_base.copy()

    for operand_value in np.unique(
        operand_perm
    ):

        idx = np.where(
            operand_perm == operand_value
        )[0]

        y_shuffled[idx] = rng.permutation(
            y_shuffled[idx]
        )

    # ---------------------------------
    # Repeat ALL bin-size analyses
    # ---------------------------------

    for j, bin_size in enumerate(
        bin_sizes
    ):

        acc = fixed_fold_poisson_accuracy(
            X_by_bin_size[bin_size],
            y_shuffled,
            fixed_folds,
        )

        null_accuracy_matrix[p, j] = acc

    # Highest accuracy obtained
    # anywhere in this shuffled search
    null_max_accuracy[p] = np.max(
        null_accuracy_matrix[p]
    )

    if (p + 1) % 20 == 0:
        print(
            f"Completed "
            f"{p + 1}/{n_permutations}"
        )

Completed 20/200
Completed 40/200
Completed 60/200
Completed 80/200
Completed 100/200
Completed 120/200
Completed 140/200
Completed 160/200
Completed 180/200
Completed 200/200


In [64]:
p_corrected = (
    1
    +
    np.sum(
        null_max_accuracy
        >= observed_best_accuracy
    )
) / (
    n_permutations + 1
)

print(
    "Observed best bin:",
    observed_best_bin,
)

print(
    "Observed best accuracy:",
    observed_best_accuracy,
)

print(
    "Mean null maximum:",
    null_max_accuracy.mean(),
)

print(
    "95th percentile null maximum:",
    np.percentile(
        null_max_accuracy,
        95,
    ),
)

print(
    "Corrected permutation p-value:",
    p_corrected,
)

Observed best bin: 30
Observed best accuracy: 0.12955465587044535
Mean null maximum: 0.13161943319838054
95th percentile null maximum: 0.15607287449392707
Corrected permutation p-value: 0.5522388059701493


In [65]:
per_bin_results = []

for j, bin_size in enumerate(
    bin_sizes
):

    obs = observed_accuracies[
        bin_size
    ]

    # uncorrected p-value
    p_uncorrected = (
        1
        +
        np.sum(
            null_accuracy_matrix[:, j]
            >= obs
        )
    ) / (
        n_permutations + 1
    )

    # corrected for searching
    # across all bin sizes
    p_max_corrected = (
        1
        +
        np.sum(
            null_max_accuracy
            >= obs
        )
    ) / (
        n_permutations + 1
    )

    per_bin_results.append(
        {
            "bin_size_ms": bin_size,
            "observed_accuracy": obs,
            "null_mean":
                null_accuracy_matrix[
                    :, j
                ].mean(),
            "p_uncorrected":
                p_uncorrected,
            "p_max_corrected":
                p_max_corrected,
        }
    )

permutation_df = pd.DataFrame(
    per_bin_results
)

permutation_df

,bin_size_ms,observed_accuracy,null_mean,p_uncorrected,p_max_corrected
0,30,0.129555,0.112692,0.174129,0.552239
1,60,0.099190,0.111447,0.805970,0.995025
2,90,0.111336,0.110648,0.477612,0.950249
3,100,0.109312,0.110192,0.532338,0.970149
4,150,0.089069,0.110962,0.940299,1.000000
5,180,0.101215,0.112267,0.731343,0.995025
6,225,0.080972,0.111225,0.970149,1.000000
7,300,0.076923,0.112267,1.000000,1.000000
8,450,0.087045,0.110698,0.950249,1.000000
9,900,0.091093,0.110304,0.925373,1.000000
